> **Note (2026-09-25).** Outputs were stripped from this notebook. The run it originally recorded executed cells out of order on Python 3.14.2 with unrecorded package versions, so its figures could not be tied to its metrics (`docs/review/panel-repro.md`, N1-N3). The authoritative numbers are regenerated by `python scripts/reproduce.py` and recorded in `docs/review/REPRODUCTION.md` and `docs/EXPERIMENTS.md`. Run this notebook top to bottom in the pinned environment (`requirements.txt`) if you want the walkthrough.


# Wildfire Duration Prediction
**Objective**: Predict how long a fire will burn based on initial discovery conditions.

This notebook demonstrates the end-to-end workflow, using the modular code from the `src` directory.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import logging

# Configure logging to see output in the notebook
# We use force=True to ensure the logs are displayed even if Jupyter has already configured logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s', force=True)

# Add the parent directory to the path so we can import from 'src'
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_loader import load_data
from src.features import preprocess_data
from src.train_model import train_model

## 1. Load Data
We load the data from the local SQLite database. We filter for recent years (2010+) to ensure relevance.

In [ ]:
# The database is not committed; run scripts/download_data.py first (see README)
db_path = '../data/FPA_FOD_20221014.sqlite'
df = load_data(db_path)
df.head()

## 2. Feature Engineering
We transform the raw data into machine-learning-ready features.

**Key Technique: Cyclical Dates**
Instead of using 'Day of Year' (1-365) linearly, we transform it into Sine and Cosine components. This helps the model understand that Day 365 (Dec 31) is temporally close to Day 1 (Jan 1).

In [ ]:
X, y = preprocess_data(df)
print("Feature Shape:", X.shape)
X.head()

## 3. Model Training & Validation
We train a **Random Forest Regressor**.

**Upgrades**:
5-Fold CV

In [ ]:
rf_model = train_model(X, y)

## 4. Evaluation & Visualization
We analyze the model using two key plots:
1. **Feature Importance**: What drives fire duration?
2. **Residual Analysis**: Where is the model making errors?

In [ ]:
from IPython.display import display, Image

print("Feature Importance:")
display(Image(filename='feature_importance.png'))

print("Residual Analysis (Predicted vs Actual):")
display(Image(filename='residuals.png'))